# 📊 Dia 2 - Storytelling com Dados

## 🎯 Objetivo
Contar histórias através dos dados usando visualizações interativas!

## 📖 Perguntas que vamos responder:
1. Qual senador mais gastou em 2022?
2. Quais os tipos de despesas mais comuns?
3. Como foi a evolução dos gastos ao longo do ano?
4. Qual a distribuição de valores das despesas?
5. Quais fornecedores receberam mais?

## 1️⃣ Importar Bibliotecas

In [ ]:
# Bibliotecas básicas
import pandas as pd
import numpy as np

# Biblioteca de visualização INTERATIVA
import plotly.express as px
import plotly.graph_objects as go

print("✅ Bibliotecas importadas!")
print("📊 Plotly instalado para gráficos interativos!")

## 2️⃣ Carregar e Preparar Dados

In [ ]:
# Carregar dados
df = pd.read_csv('../Dados Abertos - CEAPS - csv/despesa_ceaps_2022.csv', 
                 encoding='latin-1', 
                 sep=';',
                 skiprows=1)

# Limpar nomes das colunas
df.columns = df.columns.str.strip().str.replace('"', '')

# Converter valor para numérico (trocar vírgula por ponto)
df['VALOR'] = df['VALOR_REEMBOLSADO'].str.replace(',', '.').astype(float)

# Converter data
df['DATA'] = pd.to_datetime(df['DATA'], format='%d/%m/%Y', errors='coerce')

print(f"✅ Dados carregados: {len(df):,} despesas")
print(f"💰 Valor total: R$ {df['VALOR'].sum():,.2f}")
print(f"👥 Total de senadores: {df['SENADOR'].nunique()}")

## 📈 HISTÓRIA 1: Quem mais gastou em 2022?

In [ ]:
# Calcular total gasto por senador
gastos_senador = df.groupby('SENADOR')['VALOR'].sum().sort_values(ascending=False).head(15)

print("💰 Top 15 Senadores que MAIS GASTARAM em 2022:")
print("="*60)
for i, (senador, valor) in enumerate(gastos_senador.items(), 1):
    print(f"{i:2d}. {senador:30s} R$ {valor:>12,.2f}")
print("="*60)

In [ ]:
# GRÁFICO INTERATIVO - Top 15 Senadores
fig = px.bar(x=gastos_senador.values, 
             y=gastos_senador.index,
             orientation='h',
             title='💰 Top 15 Senadores que Mais Gastaram em 2022',
             labels={'x': 'Valor Total (R$)', 'y': 'Senador'},
             color=gastos_senador.values,
             color_continuous_scale='Reds')

fig.update_layout(height=600, showlegend=False)
fig.show()

print("\n💡 DICA: Passe o mouse sobre as barras para ver os valores exatos!")

## 📊 HISTÓRIA 2: Tipos de Despesas

In [ ]:
# Analisar tipos de despesas
tipos_despesa = df.groupby('TIPO_DESPESA')['VALOR'].agg(['sum', 'count']).sort_values('sum', ascending=False)
tipos_despesa.columns = ['Valor Total', 'Quantidade']

print("📋 Tipos de Despesas - Ranking por Valor:")
print("="*80)
for i, (tipo, row) in enumerate(tipos_despesa.head(10).iterrows(), 1):
    print(f"{i:2d}. {tipo[:50]:50s}")
    print(f"    💰 Total: R$ {row['Valor Total']:>12,.2f} | 📊 Qtd: {row['Quantidade']:>6,.0f}")
    print()

In [ ]:
# GRÁFICO DE PIZZA - Distribuição por tipo
top_tipos = tipos_despesa.head(8)

fig = px.pie(values=top_tipos['Valor Total'], 
             names=top_tipos.index,
             title='🥧 Distribuição dos Gastos por Tipo de Despesa (Top 8)',
             hole=0.4)  # Donut chart

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

print("\n💡 Clique nas legendas para mostrar/ocultar categorias!")

## 📅 HISTÓRIA 3: Evolução ao Longo do Ano

In [ ]:
# Gastos por mês
gastos_mes = df.groupby('MES')['VALOR'].sum().sort_index()

meses_nome = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 
              'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

print("📅 Gastos Mensais em 2022:")
print("="*50)
for mes, valor in gastos_mes.items():
    print(f"{meses_nome[int(mes)-1]:3s}: R$ {valor:>12,.2f}")
print("="*50)
print(f"TOTAL: R$ {gastos_mes.sum():>12,.2f}")

In [ ]:
# GRÁFICO DE LINHA - Evolução mensal
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[meses_nome[int(m)-1] for m in gastos_mes.index],
    y=gastos_mes.values,
    mode='lines+markers',
    name='Gastos',
    line=dict(color='#FF6B6B', width=3),
    marker=dict(size=10)
))

fig.update_layout(
    title='📈 Evolução dos Gastos ao Longo de 2022',
    xaxis_title='Mês',
    yaxis_title='Valor Total (R$)',
    hovermode='x unified',
    height=500
)

fig.show()

print("\n💡 Passe o mouse sobre os pontos para ver os valores!")

## 💵 HISTÓRIA 4: Distribuição de Valores

In [ ]:
# Estatísticas dos valores
print("💵 Estatísticas das Despesas:")
print("="*50)
print(f"Menor despesa:  R$ {df['VALOR'].min():>12,.2f}")
print(f"Maior despesa:  R$ {df['VALOR'].max():>12,.2f}")
print(f"Média:          R$ {df['VALOR'].mean():>12,.2f}")
print(f"Mediana:        R$ {df['VALOR'].median():>12,.2f}")
print("="*50)

# Faixas de valores
print("\n📊 Distribuição por Faixa de Valor:")
faixas = [
    (0, 500, 'Até R$ 500'),
    (500, 1000, 'R$ 500 - R$ 1.000'),
    (1000, 2000, 'R$ 1.000 - R$ 2.000'),
    (2000, 5000, 'R$ 2.000 - R$ 5.000'),
    (5000, 10000, 'R$ 5.000 - R$ 10.000'),
    (10000, float('inf'), 'Acima de R$ 10.000')
]

for min_val, max_val, label in faixas:
    count = len(df[(df['VALOR'] >= min_val) & (df['VALOR'] < max_val)])
    pct = (count / len(df)) * 100
    print(f"{label:25s}: {count:>6,} despesas ({pct:>5.1f}%)")

In [ ]:
# HISTOGRAMA - Distribuição de valores
fig = px.histogram(df[df['VALOR'] < 10000],  # Filtrar valores muito altos para melhor visualização
                   x='VALOR',
                   nbins=50,
                   title='📊 Distribuição dos Valores das Despesas (até R$ 10.000)',
                   labels={'VALOR': 'Valor da Despesa (R$)', 'count': 'Quantidade'},
                   color_discrete_sequence=['#4ECDC4'])

fig.update_layout(height=500)
fig.show()

print("\n💡 A maioria das despesas está concentrada em valores menores!")

## 🏢 HISTÓRIA 5: Principais Fornecedores

In [ ]:
# Top fornecedores
top_fornecedores = df.groupby('FORNECEDOR')['VALOR'].sum().sort_values(ascending=False).head(15)

print("🏢 Top 15 Fornecedores que Mais Receberam:")
print("="*70)
for i, (fornecedor, valor) in enumerate(top_fornecedores.items(), 1):
    print(f"{i:2d}. {fornecedor[:40]:40s} R$ {valor:>12,.2f}")
print("="*70)

In [ ]:
# GRÁFICO DE BARRAS - Top Fornecedores
fig = px.bar(x=top_fornecedores.index, 
             y=top_fornecedores.values,
             title='🏢 Top 15 Fornecedores - Total Recebido em 2022',
             labels={'x': 'Fornecedor', 'y': 'Valor Total (R$)'},
             color=top_fornecedores.values,
             color_continuous_scale='Blues')

fig.update_layout(height=600, xaxis_tickangle=-45, showlegend=False)
fig.show()

print("\n💡 Gire o gráfico e dê zoom para explorar melhor!")

## 🎯 RESUMO FINAL - A História Completa

In [ ]:
print("="*80)
print("📊 RESUMO EXECUTIVO - CEAPS 2022")
print("="*80)
print()
print(f"💰 VALOR TOTAL GASTO: R$ {df['VALOR'].sum():,.2f}")
print(f"📊 TOTAL DE DESPESAS: {len(df):,}")
print(f"👥 SENADORES ATIVOS: {df['SENADOR'].nunique()}")
print(f"🏢 FORNECEDORES ÚNICOS: {df['FORNECEDOR'].nunique():,}")
print()
print("🔝 DESTAQUES:")
print(f"   • Senador que mais gastou: {gastos_senador.index[0]}")
print(f"     Valor: R$ {gastos_senador.values[0]:,.2f}")
print()
print(f"   • Tipo de despesa mais comum: {tipos_despesa.index[0][:50]}")
print(f"     Total: R$ {tipos_despesa.iloc[0]['Valor Total']:,.2f}")
print()
print(f"   • Mês com maior gasto: {meses_nome[gastos_mes.idxmax()-1]}")
print(f"     Valor: R$ {gastos_mes.max():,.2f}")
print()
print(f"   • Média por despesa: R$ {df['VALOR'].mean():,.2f}")
print("="*80)

## 🎓 Conclusões e Insights

### 📌 O que descobrimos:

1. **Concentração de Gastos**: Poucos senadores concentram a maior parte dos gastos
2. **Tipos de Despesa**: Passagens aéreas e divulgação são as categorias mais comuns
3. **Sazonalidade**: Há variação nos gastos ao longo do ano
4. **Distribuição**: A maioria das despesas são de valores menores (até R$ 2.000)
5. **Fornecedores**: Companhias aéreas e empresas de comunicação dominam

### 💡 Próximos Passos:
- Comparar com anos anteriores
- Analisar por partido político
- Identificar padrões suspeitos
- Criar dashboard interativo